    Instructions:
    1. Run task_0a.py to generate the vector database if not already generated.
    2. Press Run All (or restart kernel and run all cells).
    3. You will be prompted to provide input values.
    
    Task 5 (LS3): Implement a program which, (a) given one of the feature models and (b) a value k,
    – creates (and saves) a label-label similarity matrix,
    – performs a user selected dimensionality reduction technique (SVD, NNMF, LDA, k-means) on this label-label
    similarity matrix,
    – stores the latent semantics in a properly named output file
    – lists label-weight pairs, ordered in decreasing order of weights

In [1]:
FEATURE_SPACE = input("Provide a feature space [color, hog, avgpool, layer3, fc, resnet_output].")

DIM_REDUCTION = input("Provide a dimensionality reduction technique [svd, nnmf, lda, kmeans].")

K = int(input("Enter K, the top K latent semantics to extract for the selected feature space."))


In [2]:
from utils.database_utils import retrieve
feature_vectors = retrieve(f'{FEATURE_SPACE}.pt')

print("Generating top-", K, " label latent semantics under ", FEATURE_SPACE, " feature space using: ", DIM_REDUCTION)

Generating top- 10  label latent semantics under  color  feature space using:  svd


In [3]:
from feature_models.feature_matrix.label_label_similarity import LabelLabelSimilarity
from utils.dataset_utils import initialize_dataset

label_similarity_generator = LabelLabelSimilarity(feature_vectors, initialize_dataset().categories)
label_feature_vectors = label_similarity_generator.get_matrix()

print("Label-label similarity matrix for faces as example:\n")
print(label_feature_vectors[0])
print("Shape: ", label_feature_vectors[0][1].shape)

Files already downloaded and verified
Label-label similarity matrix for faces as example:

('Faces', array([1.        , 0.98369044, 0.96256459, 0.93293577, 0.91044587,
       0.95941782, 0.9801448 , 0.9782958 , 0.97739536, 0.98499173,
       0.97445691, 0.95479637, 0.96176189, 0.98671639, 0.97349226,
       0.97667837, 0.98306686, 0.95382631, 0.96281797, 0.9819755 ,
       0.9833129 , 0.93000686, 0.97684109, 0.98437691, 0.98035264,
       0.98465395, 0.9763028 , 0.97611195, 0.98655921, 0.97934878,
       0.98223162, 0.98663467, 0.97147089, 0.95852095, 0.97833741,
       0.97489709, 0.97073072, 0.98183101, 0.98193359, 0.98591638,
       0.97293055, 0.98651278, 0.9759537 , 0.96183228, 0.98358393,
       0.97179103, 0.94233453, 0.95925617, 0.96887547, 0.98206317,
       0.97108442, 0.9855454 , 0.93895221, 0.95758969, 0.98065752,
       0.96918756, 0.97858036, 0.96705568, 0.98187315, 0.96881986,
       0.97229779, 0.9768458 , 0.98326385, 0.98064798, 0.94031328,
       0.87757593, 0.9776742

In [4]:
if DIM_REDUCTION == "svd":
    from feature_reducers.svd import SVDReducer
    reducer = SVDReducer


elif DIM_REDUCTION == "nnmf":
    from feature_reducers.nnmf import NNMFReducer
    reducer = NNMFReducer

elif DIM_REDUCTION == "lda":
    from feature_reducers.lda import LDAReducer
    reducer = LDAReducer

else:
    # kmeans.
    from feature_reducers.kmeans import KMeansReducer
    reducer = KMeansReducer

reducer = reducer(label_feature_vectors, K)

similarity_matrix = reducer.get_similarity_matrix(label_feature_vectors)

latent_semantics = reducer.reduce_features(label_feature_vectors)
print("Top K latent semantics: ")
print(latent_semantics)
print("Shape: ", latent_semantics.shape)

Top K latent semantics: 
[[ 9.72943370e+00 -6.62209407e-02 -4.77198996e-02 ...  1.99344905e-02
   1.83025523e-03  8.90454918e-03]
 [ 9.47152373e+00 -8.94088338e-02 -1.18199676e-01 ...  1.95944515e-02
   4.68319355e-03  2.47010610e-02]
 [ 9.48704924e+00 -1.08773970e-01 -4.19673944e-02 ...  1.20558837e-02
   1.51630971e-02 -3.19560793e-02]
 ...
 [ 9.71647727e+00  7.53694774e-02  1.73277705e-02 ...  1.32701451e-02
   1.34808367e-02  6.59884679e-03]
 [ 9.71156890e+00  7.16078967e-02  9.59502839e-04 ... -6.27467699e-03
  -1.13517138e-02  2.64013604e-03]
 [ 9.34505222e+00  6.51127408e-02  1.36529965e-02 ... -2.13070053e-02
  -1.20474433e-02  1.17901595e-02]]
Shape:  (101, 10)


In [5]:
# Store the latent semantics in a properly named file.
# We opt to store just the reducer, as we anyway can generate the latent space quickly
# by loading the feature space and passing it to the reducer, eg:
#
# unpicked_reducer = retrieve(f'LS3_color_svd_reducer.pt')
# feature_vectors = retrieve(f'color.pt')
#
# unpickled_reducer.reduce_features(feature_vectors)

from utils.database_utils import store

store(reducer, f'LS3_{FEATURE_SPACE}_{DIM_REDUCTION}_reducer.pt')


 Saving:  LS3_color_svd_reducer.pt 



In [7]:
# List label-weight pairs, ordered in decreasing order of weights

# We are to showcase which labels contribute more to each latent feature.
# This is taking the object-feature factor matrix, and sorting by each
# latent feature's weight.

labels = [feature_tuple[0] for feature_tuple in label_feature_vectors.values()]

image_weight_tuples = list(zip(labels, similarity_matrix))

print("Label - weight pairs sorted in descending order of weights for each latent feature:")

for i in range(K):
    print("\n\nLatent feature: ", i + 1)
    for LABEL, weight in sorted(image_weight_tuples, key=lambda x : x[1][i], reverse=True):
        print("(Label: ", LABEL, ", Weight: ", weight[i], end="),\t")

Label - weight pairs sorted in descending order of weights for each latent feature:


Latent feature:  1
(Label:  starfish , Weight:  0.1010710082369556),	(Label:  cup , Weight:  0.10104786675324169),	(Label:  scorpion , Weight:  0.10104454500371468),	(Label:  crocodile , Weight:  0.10099572381044863),	(Label:  ceiling_fan , Weight:  0.10096109081000725),	(Label:  llama , Weight:  0.10095972534875489),	(Label:  emu , Weight:  0.10093264991502804),	(Label:  butterfly , Weight:  0.10089782784432248),	(Label:  chandelier , Weight:  0.10089385841429993),	(Label:  rhino , Weight:  0.10088564065435555),	(Label:  crab , Weight:  0.10088229518299209),	(Label:  flamingo , Weight:  0.10083210299727294),	(Label:  chair , Weight:  0.10082704048585174),	(Label:  ewer , Weight:  0.10082384514660488),	(Label:  crayfish , Weight:  0.10081091456182818),	(Label:  euphonium , Weight:  0.10080944965217947),	(Label:  cougar_face , Weight:  0.10080347298389217),	(Label:  barrel , Weight:  0.1007955146800303